# Chapter 3 — Spherical orbits with AGAMA

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/03_spherical_orbits.ipynb)

**Goal.** Understand the shape of orbits in spherical galaxies — the rosettes of Chapter 2 — through
two numbers: the **radial period** $T_r$ and the **apsidal angle** $\Delta\phi$. Compute them three
ways (by hand, from an integrated orbit, and with AGAMA) and learn to use AGAMA for many orbits at
once and in physical units.

**You will learn**
1. to integrate many orbits in one call to `agama.orbit`;
2. the radial period and apsidal angle as integrals over the effective potential;
3. AGAMA's orbital **frequencies** $\Omega_r$, $\Omega_\phi$ and how they encode $T_r$ and $\Delta\phi$;
4. why $\pi \le \Delta\phi \le 2\pi$ in galaxies, with the isochrone as an exact test;
5. how to move from $G = 1$ units to kpc, km/s and $M_\odot$ — and why `setUnits` must come first.

**Units.** Sections 3.1–3.4: $G = 1$, Plummer sphere with $M = b = 1$ (AGAMA without `setUnits`,
as in Chapters 1–2). Section 3.5 switches to physical units.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq

G, M, b = 1.0, 1.0, 1.0
pot = agama.Potential(type='Plummer', mass=M, scaleRadius=b)

def phi_plummer(r):
    return -G * M / np.sqrt(r**2 + b**2)

def vc_plummer(r):
    return np.sqrt(G * M * r**2 / (r**2 + b**2)**1.5)

## 3.1 Many orbits at once

In Chapter 2 we launched one star from $r = 1$ moving tangentially at 70% of the circular speed.
Here we launch five, at 20%, 40%, 60%, 80% and 100% of $v_c$, to see how the orbit shape changes with
angular momentum.

`agama.orbit` accepts an array of initial conditions of shape `(N, 6)` and integrates all $N$ orbits
in one call (in parallel, on all CPU cores, if AGAMA was built with OpenMP). The output format then
changes: instead of a pair `(times, trajectory)`, it returns an array of shape `(N, 2)` whose row `i`
holds the times and the trajectory of orbit `i`.

In [ ]:
fractions = np.array([0.2, 0.4, 0.6, 0.8, 1.0])          # tangential speed / circular speed
n_orb = len(fractions)
ic = np.zeros((n_orb, 6))                                # one row (x, y, z, vx, vy, vz) per orbit
ic[:, 0] = 1.0                                           # all start at x = 1 ...
ic[:, 4] = fractions * vc_plummer(1.0)                   # ... moving in +y

orbits = agama.orbit(potential=pot, ic=ic, time=40.0, trajsize=4001, dtype=float, verbose=False)
print('output shape:', orbits.shape, '-> orbits[i, 0] = times, orbits[i, 1] = trajectory of shape',
      orbits[0, 1].shape)

fig, ax = plt.subplots(1, n_orb, figsize=(16, 3.5))
for i in range(n_orb):
    traj = orbits[i, 1]
    ax[i].plot(traj[:, 0], traj[:, 1], lw=0.5)
    ax[i].plot(0, 0, 'k*')
    ax[i].set_aspect('equal'); ax[i].set_xlim(-1.1, 1.1); ax[i].set_ylim(-1.1, 1.1)
    ax[i].set_title(f'$v = {fractions[i]:.1f}\\,v_c$')
plt.tight_layout(); plt.show()

**Figure 3.1.** Five orbits in the Plummer sphere launched tangentially at $r = 1$ with 0.2, 0.4, 0.6, 0.8 and 1.0 times the circular speed (left to right), integrated for 40 time units, in the $x$–$y$ plane.

At low angular momentum (Figure 3.1, left) the star plunges close to the centre and the rosette has narrow petals; at
$v = v_c$ the orbit is a circle (right). In between, every orbit fills an annulus between its pericentre
and apocentre. To describe a rosette we need two numbers:

- the **radial period** $T_r$: the time from one apocentre to the next;
- the **apsidal angle** $\Delta\phi$: how far the star advances in azimuth during $T_r$.

If $\Delta\phi / 2\pi$ is a rational number the orbit eventually closes; otherwise it never repeats
and fills the annulus densely.

## 3.2 Radial period and apsidal angle — three ways

### By hand

From Chapter 1, in the effective potential the radial speed is
$v_r = \sqrt{2[E - \Phi(r)] - L^2/r^2}$, and the azimuthal angle grows at the rate
$d\phi/dt = L/r^2$, where $E$ and $L$ are the energy and angular momentum per unit mass, $\Phi(r)$ the potential,
and $\phi$ the azimuth in the plane of the orbit. Going from pericentre $r_p$ to apocentre $r_a$ and back,

$$ T_r = 2\int_{r_p}^{r_a} \frac{dr}{v_r}, \qquad
   \Delta\phi = 2\int_{r_p}^{r_a} \frac{L}{r^2}\,\frac{dr}{v_r} . $$

Both integrands blow up at the turning points, where $v_r = 0$ (the integrals are finite, but
numerically awkward). The substitution $r = \bar r + \delta\sin\theta$, with
$\bar r = (r_a + r_p)/2$ and $\delta = (r_a - r_p)/2$, removes the problem: $v_r$ vanishes like
$\cos\theta$ at both ends, and so does $dr = \delta\cos\theta\, d\theta$, so the ratio stays finite.

In [ ]:
def radial_period_and_apsidal_angle(E, L, rp, ra):
    """T_r and Delta phi by direct integration over the effective potential (theta substitution)."""
    rbar, delta = 0.5 * (ra + rp), 0.5 * (ra - rp)
    def vr(r):
        # max(..., 0) guards against tiny negative values from round-off right at the turning points
        return np.sqrt(np.maximum(2 * (E - phi_plummer(r)) - L**2 / r**2, 0))
    def dr_over_vr(theta):                       # (dr/dtheta) / v_r, finite at theta = +-pi/2
        r = rbar + delta * np.sin(theta)
        return delta * np.cos(theta) / vr(r)
    Tr = 2 * quad(dr_over_vr, -np.pi / 2, np.pi / 2)[0]
    dphi = 2 * quad(lambda th: L / (rbar + delta * np.sin(th))**2 * dr_over_vr(th), -np.pi / 2, np.pi / 2)[0]
    return Tr, dphi

# the orbit of Chapter 2: start at r = 1, tangential, 70% of v_c
r0, v0 = 1.0, 0.7 * vc_plummer(1.0)
E, L = 0.5 * v0**2 + phi_plummer(r0), r0 * v0
rp, ra = pot.Rperiapo([E, L])                    # turning points (Chapter 1 §1.5)
Tr_hand, dphi_hand = radial_period_and_apsidal_angle(E, L, rp, ra)
print(f'by hand:     T_r = {Tr_hand:.8f},  apsidal angle = {dphi_hand:.8f} rad')

### From the integrated orbit

As in Chapter 2: find successive apocentres (local maxima of $r$) and measure the time and the
azimuth between them. With a finite output interval $\Delta t$ the maximum is located only to within
$\Delta t$, so we store the orbit densely ($\Delta t = 10^{-3}$).

In [ ]:
xv0 = [r0, 0, 0, 0, v0, 0]
t, traj = agama.orbit(potential=pot, ic=xv0, time=60.0, trajsize=60001, dtype=float, verbose=False)
r = np.linalg.norm(traj[:, :3], axis=1)
phi = np.unwrap(np.arctan2(traj[:, 1], traj[:, 0]))       # continuous azimuth (see Chapter 2 §2.6)
i_apo = np.where((r[1:-1] > r[:-2]) & (r[1:-1] >= r[2:]))[0] + 1   # indices of apocentres
Tr_orbit = np.diff(t[i_apo]).mean()
dphi_orbit = np.diff(phi[i_apo]).mean()
print(f'from orbit:  T_r = {Tr_orbit:.8f},  apsidal angle = {dphi_orbit:.8f} rad   ({len(i_apo)} apocentres)')

### With AGAMA's frequencies

A regular orbit in a spherical potential is a combination of two periodic motions: the radial
oscillation with frequency $\Omega_r = 2\pi/T_r$, and the azimuthal motion with mean angular speed
$\Omega_\phi = \Delta\phi / T_r$. So

$$ T_r = \frac{2\pi}{\Omega_r}, \qquad \Delta\phi = 2\pi\,\frac{\Omega_\phi}{\Omega_r} . $$

`agama.ActionFinder(pot)` builds an object that, called on a phase-space point with
`frequencies=True`, returns the **actions** $(J_r, J_z, J_\phi)$ (radial, vertical and azimuthal; for a spherical potential
$J_\phi = L_z$ and $J_z = L - |L_z|$) and the **frequencies**
$(\Omega_r, \Omega_z, \Omega_\phi)$ — each a triplet, or an `(N, 3)` array for $N$ points.
(Actions are the subject of Chapter 9; here we only need the frequencies. For a planar orbit in a
spherical potential $\Omega_z = \Omega_\phi$.) No orbit integration is needed: AGAMA computes them
from $E$ and $L$ with integrals like ours.

In [ ]:
af = agama.ActionFinder(pot)
J, Omega = af(xv0, frequencies=True)              # two triplets: (Jr, Jz, Jphi), (Omega_r, Omega_z, Omega_phi)
Tr_agama = 2 * np.pi / Omega[0]
dphi_agama = 2 * np.pi * Omega[2] / Omega[0]
print(f'AGAMA:       T_r = {Tr_agama:.8f},  apsidal angle = {dphi_agama:.8f} rad')

assert abs(Tr_orbit / Tr_hand - 1) < 1e-4 and abs(dphi_orbit / dphi_hand - 1) < 1e-4
assert abs(Tr_agama / Tr_hand - 1) < 1e-6 and abs(dphi_agama / dphi_hand - 1) < 1e-6

plt.figure(figsize=(7, 3.5))
plt.plot(t, r, lw=0.8)
plt.plot(t[i_apo], r[i_apo], 'o', color='C3', label='apocentres')
for k in range(1, 5):
    plt.axvline(k * Tr_agama, color='gray', ls=':', lw=0.8)
plt.xlim(0, 25); plt.xlabel('time'); plt.ylabel('$r$')
plt.title('dotted lines: multiples of $T_r = 2\\pi/\\Omega_r$ from AGAMA'); plt.legend(); plt.show()

**Figure 3.2.** Radius against time for the orbit of §3.2 (launched at $0.7\,v_c$), with the apocentres found on the orbit (red points) and multiples of AGAMA's radial period $T_r = 2\pi/\Omega_r$ (dotted lines).

All three methods agree: the hand integral and AGAMA to better than $10^{-6}$, the orbit (Figure 3.2) to about
$10^{-4}$ (set by how finely the apocentres are sampled). The apsidal angle, about 212°, is the 3.70 rad
measured in Chapter 2.

## 3.3 An exact test: the isochrone

The **isochrone** potential (Hénon 1959),

$$ \Phi(r) = -\frac{GM}{b + \sqrt{b^2 + r^2}}, $$

with total mass $M$ and scale radius $b$, is the most general spherical potential in which orbits can be found in closed form. Its radial
period and apsidal angle are

$$ T_r = \frac{2\pi G M}{(-2E)^{3/2}}, \qquad
   \Delta\phi = \pi\left(1 + \frac{L}{\sqrt{L^2 + 4GMb}}\right) , $$

for an orbit of energy $E < 0$ and angular momentum $L$ (per unit mass).

Like Kepler's third law, $T_r$ depends only on energy. The apsidal angle runs from $\pi$ for radial
orbits ($L \to 0$) to $2\pi$ for large $L$. We check AGAMA's frequencies against these formulae for
100 orbits at once: `ActionFinder` accepts an `(N, 6)` array and returns `(N, 3)` arrays.

In [ ]:
pot_iso = agama.Potential(type='Isochrone', mass=M, scaleRadius=b)
af_iso = agama.ActionFinder(pot_iso)

rng = np.random.default_rng(1)                   # random-number generator with a fixed seed (reproducible)
n = 100
xv = np.zeros((n, 6))
xv[:, 0] = rng.uniform(0.1, 5, n)                # start at x between 0.1 and 5 ...
vc_iso = np.sqrt(-xv[:, 0] * pot_iso.force(xv[:, :3])[:, 0])        # circular speed there (Chapter 0)
xv[:, 3] = rng.uniform(-0.5, 0.5, n) * vc_iso    # ... with random radial ...
xv[:, 4] = rng.uniform(0.05, 1.0, n) * vc_iso    # ... and tangential velocities (all bound orbits)

J, Omega = af_iso(xv, frequencies=True)
E_iso = pot_iso.potential(xv[:, :3]) + 0.5 * np.sum(xv[:, 3:]**2, axis=1)
L_iso = np.abs(xv[:, 0] * xv[:, 4])              # |x v_y - y v_x| with y = 0

Tr_exact = 2 * np.pi * G * M / (-2 * E_iso)**1.5
dphi_exact = np.pi * (1 + L_iso / np.sqrt(L_iso**2 + 4 * G * M * b))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].loglog(Tr_exact, 2 * np.pi / Omega[:, 0], 'o', ms=4)
ax[0].plot([Tr_exact.min(), Tr_exact.max()], [Tr_exact.min(), Tr_exact.max()], 'k-', lw=0.8)
ax[0].set_xlabel('$T_r$ formula'); ax[0].set_ylabel(r'$T_r = 2\pi/\Omega_r$ (AGAMA)')
ax[1].plot(L_iso, 2 * np.pi * Omega[:, 2] / Omega[:, 0] / np.pi, 'o', ms=4, label='AGAMA')
Lg = np.linspace(0, L_iso.max(), 200)
ax[1].plot(Lg, 1 + Lg / np.sqrt(Lg**2 + 4 * G * M * b), 'k-', label='formula')
ax[1].set_xlabel('$L$'); ax[1].set_ylabel(r'$\Delta\phi / \pi$'); ax[1].legend()
plt.tight_layout(); plt.show()

err_T = np.max(np.abs(2 * np.pi / Omega[:, 0] / Tr_exact - 1))
err_phi = np.max(np.abs(2 * np.pi * Omega[:, 2] / Omega[:, 0] / dphi_exact - 1))
print(f'max relative error over {n} isochrone orbits: T_r {err_T:.1e}, apsidal angle {err_phi:.1e}')
assert err_T < 1e-8 and err_phi < 1e-8

**Figure 3.3.** 100 random orbits in the isochrone potential. Left: radial period from AGAMA's frequencies against the formula (log–log; the line marks equality). Right: apsidal angle $\Delta\phi/\pi$ against angular momentum $L$, from AGAMA (points) and the formula (line).

AGAMA reproduces the isochrone to machine precision (Figure 3.3) — it has the exact isochrone formulae built in.
For other potentials, such as the Plummer sphere of §3.2, it evaluates the integrals numerically;
there it agreed with our hand integration to better than $10^{-6}$.

## 3.4 Why $\pi \le \Delta\phi \le 2\pi$

Two potentials have closed orbits for every $E$ and $L$:

- **Harmonic** potential, $\Phi \propto r^2$ (the potential *inside* a homogeneous sphere, Chapter 1):
  orbits are ellipses *centred* on the origin. The star reaches apocentre twice per revolution, so
  $\Delta\phi = \pi$.
- **Kepler** potential, $\Phi \propto -1/r$ (a point mass): orbits are ellipses with the origin at a
  *focus*. One apocentre per revolution, so $\Delta\phi = 2\pi$.

A galaxy is in between: its core is nearly homogeneous, and far outside it all the mass looks like a
point. So we expect $\Delta\phi \to \pi$ for orbits deep in the core and $\Delta\phi \to 2\pi$ for
orbits far outside. (For any spherical system whose density does not increase outwards,
$\pi \le \Delta\phi \le 2\pi$; see Binney & Tremaine 2008, §3.1.)

We compute $\Delta\phi$ for 300 Plummer orbits with apocentres from $0.01\,b$ to $1000\,b$, all
launched tangentially at 70% of the local circular speed (same shape as §3.2, different size).

In [ ]:
r_apo = np.logspace(-2, 3, 300)
xv = np.zeros((len(r_apo), 6))
xv[:, 0] = r_apo
xv[:, 4] = 0.7 * vc_plummer(r_apo)
J, Omega = af(xv, frequencies=True)
dphi = 2 * np.pi * Omega[:, 2] / Omega[:, 0]

plt.figure(figsize=(7, 4))
plt.semilogx(r_apo / b, dphi / np.pi)
plt.axhline(1, color='gray', ls=':'); plt.axhline(2, color='gray', ls=':')
plt.text(0.012, 1.05, 'harmonic limit'); plt.text(100, 1.92, 'Kepler limit')
plt.axvline(ra, color='C3', ls='--', lw=0.8, label='orbit of §3.2')
plt.xlabel('apocentre $r_a / b$'); plt.ylabel(r'apsidal angle $\Delta\phi / \pi$'); plt.legend()
plt.title('Plummer sphere, orbits launched at $0.7\\,v_c$'); plt.show()

print(f'apsidal angle / pi: at r_a = 0.01 b -> {dphi[0] / np.pi:.5f},  at r_a = 1000 b -> {dphi[-1] / np.pi:.5f}')
assert abs(dphi[0] / np.pi - 1) < 1e-3 and abs(dphi[-1] / np.pi - 2) < 1e-2
assert np.all((dphi >= np.pi * (1 - 1e-6)) & (dphi <= 2 * np.pi * (1 + 1e-6)))

**Figure 3.4.** Apsidal angle $\Delta\phi/\pi$ of 300 Plummer orbits launched tangentially at $0.7\,v_c$, against apocentre $r_a/b$. Dotted lines: the harmonic (1) and Kepler (2) limits; dashed line: the orbit of §3.2.

In Figure 3.4 the apsidal angle rises smoothly from the harmonic limit in the core to the Kepler limit in the
outskirts, and the transition happens around $r \sim b$, where the density changes from flat to
steeply falling. Stars in real galaxies — including the Sun — move on rosettes, not ellipses.

## 3.5 Physical units

So far $G = M = b = 1$. To turn these numbers into kpc, km/s and years, pick a mass scale $M_0$ and a
length scale $L_0$. Because $G = 1$ in our units, the velocity and time scales follow:

$$ V_0 = \sqrt{\frac{G M_0}{L_0}}, \qquad T_0 = \frac{L_0}{V_0} . $$

Every dimensionless result converts by multiplying by the right combination: positions by $L_0$,
velocities by $V_0$, times by $T_0$, energies by $V_0^2$, angular momenta by $L_0 V_0$; angles stay
the same. For a Plummer bulge with $M_0 = 10^{10}\,M_\odot$ and $L_0 = b = 1$ kpc:

In [ ]:
G_phys = 4.30091727067736e-6          # G in kpc (km/s)^2 / Msun
KPC_PER_KMS_IN_MYR = 977.7922216807891  # 1 kpc / (1 km/s) in Myr = 3.0857e16 km / (1 km/s) / 3.1558e13 s
M0, L0 = 1e10, 1.0              # Msun, kpc
V0 = np.sqrt(G_phys * M0 / L0)  # km/s
T0_Myr = L0 / V0 * KPC_PER_KMS_IN_MYR
print(f'V0 = {V0:.2f} km/s,  T0 = {T0_Myr:.4f} Myr')
print(f'orbit of §3.2: apocentre {ra * L0:.2f} kpc, pericentre {rp * L0:.3f} kpc, '
      f'launch speed {v0 * V0:.1f} km/s, radial period {Tr_agama * T0_Myr:.1f} Myr')

### The same orbit, computed by AGAMA in physical units

Now we ask AGAMA to work in physical units directly, with `agama.setUnits(mass=1, length=1,
velocity=1)` (1 $M_\odot$, 1 kpc, 1 km/s; Chapter 0 §0.2), and check that it gives the same orbit
after rescaling. This also demonstrates the most important rule about AGAMA's units:
**`setUnits` must be called before creating any AGAMA object.** Our `pot` was created in $G = 1$
units; watch what happens to it.

In [ ]:
Phi0_before = pot.potential([0, 0, 0])
agama.setUnits(mass=1, length=1, velocity=1)     # AGAMA prints a warning: objects already exist
print('Phi(0) of the OLD potential, before and after setUnits:', Phi0_before, pot.potential([0, 0, 0]))
print('G is now', agama.G)

The old potential now returns nonsense (it is neither $-GM/b = -1$ in model units nor
$-43009$ (km/s)$^2$ for $10^{10}\,M_\odot$ and 1 kpc). Objects created before `setUnits` must be
thrown away and **re-created**. From here on, everything is in $M_\odot$, kpc, km/s, and times are
in the unit 1 kpc/(km/s) = 977.79 Myr; `agama.getUnits()['time']` reports this unit in Myr.

In [ ]:
pot_phys = agama.Potential(type='Plummer', mass=M0, scaleRadius=L0)      # re-created in physical units
time_unit_Myr = agama.getUnits()['time']

# Conversion factors computed with AGAMA's own G, so that both calculations use identical constants.
# With L0 in kpc and V0 in km/s, T0 = L0 / V0 is already in AGAMA's time unit, 1 kpc/(km/s).
V0 = np.sqrt(agama.G * M0 / L0)
T0 = L0 / V0

# same initial conditions, converted: positions x L0, velocities x V0; total time 60 T0
xv0_phys = np.array(xv0) * np.array([L0, L0, L0, V0, V0, V0])
t_phys, traj_phys = agama.orbit(potential=pot_phys, ic=xv0_phys, time=60.0 * T0,
                                trajsize=60001, dtype=float, verbose=False)

# convert back to model units and compare with the G = 1 orbit of §3.2, point by point
pos_diff = np.max(np.abs(traj_phys[:, :3] / L0 - traj[:, :3]))
vel_diff = np.max(np.abs(traj_phys[:, 3:] / V0 - traj[:, 3:]))
print(f'max difference after rescaling: positions {pos_diff:.1e}, velocities {vel_diff:.1e} (model units)')
assert pos_diff < 1e-9 and vel_diff < 1e-9

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].plot(traj_phys[:, 0], traj_phys[:, 1], lw=0.4)
ax[0].set_aspect('equal'); ax[0].set_xlabel('x [kpc]'); ax[0].set_ylabel('y [kpc]')
ax[0].set_title(r'orbit in a $10^{10}\,M_\odot$, $b = 1$ kpc Plummer sphere')
ax[1].plot(t_phys * time_unit_Myr, np.linalg.norm(traj_phys[:, 3:], axis=1))
ax[1].set_xlim(0, 100); ax[1].set_xlabel('time [Myr]'); ax[1].set_ylabel('speed [km/s]')
plt.tight_layout(); plt.show()

**Figure 3.5.** The orbit of §3.2 recomputed in physical units, in a $10^{10}\,M_\odot$, $b = 1$ kpc Plummer sphere. Left: the orbit in the $x$–$y$ plane in kpc. Right: speed in km/s against time over the first 100 Myr.

The two calculations agree to about $10^{-12}$ (Figure 3.5 shows the orbit) — round-off: physics does not care about units, as
long as we convert consistently. *Consistently* includes the constants. When this notebook was first
written, the conversion used a time unit rounded to 977.79 Myr (an error of 2 parts in a million),
and the two orbits drifted apart by $10^{-4}$ after ten radial periods, because a small error in time
becomes a growing error in orbital phase. Take $G$ and the time unit from AGAMA (`agama.G`,
`agama.getUnits()`) or use them to full precision; never round them. In practice, choose your units once, call `setUnits` at the very top of
the notebook, and never change them.

## Summary

- `agama.orbit` integrates many orbits at once from an `(N, 6)` array of initial conditions and
  returns an `(N, 2)` array of (times, trajectory) pairs.
- A spherical orbit is a rosette described by its radial period $T_r = 2\pi/\Omega_r$ and apsidal
  angle $\Delta\phi = 2\pi\,\Omega_\phi/\Omega_r$.
- `agama.ActionFinder(pot)(xv, frequencies=True)` returns actions and frequencies without integrating
  orbits; it agrees with direct integrals and with integrated orbits, and is exact for the isochrone.
- $\Delta\phi$ goes from $\pi$ (harmonic core) to $2\pi$ (Kepler outskirts).
- Model units convert to physical ones via $V_0 = \sqrt{GM_0/L_0}$, $T_0 = L_0/V_0$. Call
  `agama.setUnits` before creating any object; objects created earlier are invalidated.

## Exercises

1. **Closed orbits.** In the Plummer sphere, find the orbit launched tangentially at $r = 1$ whose
   apsidal angle is exactly $4\pi/3$ (use `brentq` on the launch speed). Integrate it for three radial
   periods and plot it. What shape is it?
2. Use `ActionFinder` to show numerically that the isochrone radial period depends on $E$ only:
   compute $T_r$ for 100 orbits with the same $E$ and different $L$.
3. Repeat §3.4 for the Hernquist model (`type='Dehnen', gamma=1`), whose density diverges at the
   centre as $1/r$. Does $\Delta\phi$ still approach $\pi$ in the centre? Explain.
4. Repeat §3.4 for launch speeds of 30%, 70% and 95% of $v_c$ on one plot. How does $\Delta\phi$ depend
   on how eccentric the orbit is?
5. A globular cluster has $M = 10^5\,M_\odot$ and Plummer radius $b = 3$ pc. Compute $V_0$, $T_0$ and
   the radial period of the orbit of §3.2 scaled to this cluster, in Myr.
6. Time `agama.orbit` for 1, 10, 100 and 1000 orbits (`time.perf_counter()`). How does the cost scale?

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §3.1 and §3.5.2.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapter 4: https://galaxiesbook.org
- M. Hénon (1959), "L'amas isochrone: I", Annales d'Astrophysique 22, 126:
  https://ui.adsabs.harvard.edu/abs/1959AnAp...22..126H
- E. Vasiliev (2019), AGAMA, MNRAS 482, 1525: https://doi.org/10.1093/mnras/sty2672